# 从零搭建一个 LLM Agent

以 Stata 代码助手为例，基于 LangGraph + MCP + DeepSeek。

## 先理解：什么是 Agent？

Agent = LLM + 工具 + 循环决策。普通 LLM 调用是"一问一答"，Agent 是"思考->行动->观察->思考..."的循环。

## 工作流


## 第一步：State

Agent 各步骤之间传递信息的"共享内存"。对应文件：agent/state.py

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class StataAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    current_code: str
    error_info: str
    retry_count: int
    pending_action: dict

关键：Annotated[list, add_messages] 让消息自动追加而非覆盖。TypedDict 给每个字段定义类型。

## 第二步：LLM 客户端

双模型架构：coder(非思考模式)快速生成代码，reviewer(思考模式)深度分析错误。对应文件：llm/client.py

In [ ]:
from openai import AsyncOpenAI

class DualLLMClient:
    def __init__(self):
        self.client = AsyncOpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
    async def call_coder(self, messages):
        return await self.client.chat.completions.create(model=CODER_MODEL, messages=messages, extra_body={"thinking":{"type":"disabled"}})
    async def call_reviewer(self, messages):
        return await self.client.chat.completions.create(model=REVIEWER_MODEL, messages=messages, extra_body={"thinking":{"type":"enabled"}})

关键：使用 AsyncOpenAI 而非 OpenAI，因为 LangGraph 的 ainvoke 是异步的。

## 第三步：工具

LLM 只会"说"，工具是 Agent 的"手"。两类工具：MCP工具(stata_run, get_log) + 文件工具(read_file, create_do_file, modify_do_file)

In [ ]:
from langchain_core.tools import tool
from pathlib import Path

@tool
async def create_do_file(path: str, content: str) -> str:
    """创建新的 Stata .do 文件"""
    file_path = Path(path)
    if file_path.exists():
        return f"Error: 文件已存在: {file_path}"
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding="utf-8-sig")
    return f"文件已创建: {file_path}"

关键：@tool 装饰器自动把函数签名和 docstring 转换成 LLM 能理解的工具描述。ToolNode 自动解析 tool_calls 并执行对应函数。

## 第四步：Prompt

Agent 的"操作手册"。LLM 看 Prompt 知道有什么工具、怎么用、输出什么格式。对应文件：agent/prompts.py

我们设计了 [ACTION:xxx] 指令协议：LLM 输出 [ACTION:create_do_file] 这样的指令，解析器识别后路由到正确工具。这比 function calling 更可控。

## 第五步：节点

节点是工作流中每一步具体做什么。4 个节点：generate(生成代码) -> prepare_tools(封装工具调用) -> execute_tools(执行工具) -> review(错误修正)

In [ ]:
async def generate_code(state, llm_client) -> dict:
    response = await llm_client.call_coder([{"role":"system","content":CODER_SYSTEM_PROMPT},{"role":"user","content":state["messages"][-1].content}])
    code = extract_code(response)
    action = parse_action_directive(response) or {"tool":"stata_run","args":{"code":code}}
    return {"messages":[AIMessage(content=display)],"current_code":code,"pending_action":action}

关键：每个节点第一个参数是 state，返回 dict 表示对 state 的更新。functools.partial 用于注入 llm_client/tool_node 等非 state 依赖。

## 第六步：条件边

Agent 最核心的能力：根据执行结果做不同的事。对应文件：agent/edges.py

In [ ]:
def should_continue(state) -> Literal["review","generate","__end__"]:
    for msg in reversed(state["messages"]):
        if isinstance(msg, ToolMessage):
            if "r(" in str(msg.content) and state.get("retry_count",0) < MAX_RETRY:
                return "review"      # Stata错误 + 还有重试次数 -> 修正循环
            if "读取文件成功" in str(msg.content):
                return "generate"    # 文件读取成功 -> 链式继续
            if "文件已创建" in str(msg.content) or "文件已修改" in str(msg.content):
                return END           # 文件写操作 -> 结束
            return END               # 执行成功 -> 结束
    return END

关键：条件边返回的是"下一站的名字"而非状态更新。reversed(messages) 从后往前找最近的 ToolMessage。常见Bug：路由逻辑覆盖不全。

## 第七步：拼装图

有了节点和边，组装成完整的 StateGraph。对应文件：agent/graph.py

In [ ]:
from functools import partial
from langgraph.graph import StateGraph, END

class StataAgent:
    def _build_graph(self):
        workflow = StateGraph(StataAgentState)
        workflow.add_node("generate", partial(generate_code, llm_client=self.llm))
        workflow.add_node("prepare_tools", prepare_tool_call)
        workflow.add_node("execute_tools", partial(execute_tools, tool_node=self.tool_node))
        workflow.add_node("review", partial(review_error, llm_client=self.llm))
        workflow.set_entry_point("generate")
        workflow.add_edge("generate", "prepare_tools")
        workflow.add_edge("prepare_tools", "execute_tools")
        workflow.add_conditional_edges("execute_tools", should_continue, {"review":"review","generate":"generate","__end__":END})
        workflow.add_edge("review", "prepare_tools")
        return workflow.compile()

关键：StateGraph + compile() 两步走（描述+编译）。partial 注入依赖而非全局变量。固定边 add_edge vs 条件边 add_conditional_edges。

## 第八步：交互入口

启动流程：初始化 LLM -> 连接 MCP -> 构建 Agent -> 交互循环。对应文件：main.py

调试策略：1) 每条消息都打印 2) 扫描所有 ToolMessage 判断最终状态 3) 保留完整错误信息供 review 使用。

---

## 五个核心设计模式

1. **ReAct模式**：思考->行动->观察->思考 循环
2. **State模式**：显式状态传递，所有节点共享
3. **Router模式**：should_continue 根据运行时结果决定路径
4. **Retry-with-Review模式**：执行->报错->LLM分析+修正->再执行
5. **Tool-as-Message模式**：工具调用和结果都是消息，LLM可理解

## 扩展练习

1. 换LLM：改 config/settings.py 的 BASE_URL 和模型名
2. 加工具：用 @tool 写函数，加入 create_tool_node
3. 改领域：换 CODER_SYSTEM_PROMPT 中的"Stata"为你的领域
4. 改流程：在 graph.py 加新节点，调整边

## 项目文件速查

| 文件 | 作用 | 何时改 |
|------|------|--------|
| agent/state.py | 状态结构 | 加字段 |
| agent/prompts.py | 提示词 | 换领域 |
| agent/nodes.py | 步骤逻辑 | 加节点 |
| agent/edges.py | 路由决策 | 加分支 |
| agent/graph.py | 拼装图 | 改流程 |
| llm/client.py | API调用 | 换模型 |
| config/settings.py | 配置 | 调参数 |
| tools/file_tools.py | 文件工具 | 加工具 |
| main.py | 入口 | 改交互 |

> 建议：第一次接触 Agent，从 agent/graph.py 开始——只55行，是整张"地图"。看懂图结构后再逐个看节点文件。